In [2]:
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import gradio as gr
from PIL import Image
import numpy as np 

In [3]:
datagen = ImageDataGenerator(rescale=1./255, validation_split=0.2)

train = datagen.flow_from_directory("train", target_size=(64, 64),
                                     batch_size=32, class_mode="binary",
                                     subset="training")

val   = datagen.flow_from_directory("train", target_size=(64, 64),
                                     batch_size=32, class_mode="binary",
                                     subset="validation")

Found 80000 images belonging to 2 classes.
Found 20000 images belonging to 2 classes.


In [6]:
model = tf.keras.Sequential([
    layers.Conv2D(32, (3,3), activation="relu", input_shape=(64, 64, 3)),
    layers.MaxPooling2D(),
    layers.Conv2D(64, (3,3), activation="relu"),
    layers.MaxPooling2D(),
    layers.Flatten(),
    layers.Dense(64, activation="relu"),
    layers.Dense(1, activation="sigmoid")   
])

model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

In [7]:
model.fit(train, validation_data=val, epochs=10)

Epoch 1/10
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 128s 51ms/step - accuracy: 0.8654 - loss: 0.3117 - val_accuracy: 0.9207 - val_loss: 0.2045
Epoch 2/10
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 98s 39ms/step - accuracy: 0.9234 - loss: 0.1960 - val_accuracy: 0.9326 - val_loss: 0.1705
Epoch 3/10
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 97s 39ms/step - accuracy: 0.9402 - loss: 0.1536 - val_accuracy: 0.9128 - val_loss: 0.2180
Epoch 4/10
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 97s 39ms/step - accuracy: 0.9522 - loss: 0.1255 - val_accuracy: 0.9370 - val_loss: 0.1665
Epoch 5/10
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 97s 39ms/step - accuracy: 0.9614 - loss: 0.1006 - val_accuracy: 0.9420 - val_loss: 0.1610
Epoch 6/10
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 99s 39ms/step - accuracy: 0.9709 - loss: 0.0769 - val_accuracy: 0.9322 - val_loss: 0.1978
Epoch 7/10
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 101s 41ms/step - accuracy: 0.9774 - loss: 0.0597 - val_accuracy: 0.9322 - val_loss: 0.2429
Epoch 8/10
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 101s 40ms/step - accuracy: 0.983

In [8]:
print(model.evaluate(val))

625/625 ━━━━━━━━━━━━━━━━━━━━ 11s 18ms/step - accuracy: 0.9437 - loss: 0.2432
[0.24317924678325653, 0.9437000155448914]


In [9]:
import gradio as gr
import numpy as np
from PIL import Image

def predict(img):
    if img is None:
        return "⚠️ Please upload an image"
    
    img_resized = img.resize((64, 64))
    arr = np.array(img_resized) / 255.0
    arr = np.expand_dims(arr, axis=0)
    score = model.predict(arr)[0][0]
    
    label = "Real" if score > 0.5 else "AI-Generated"
    confidence = score if score > 0.5 else 1 - score
    emoji = "📷" if label == "Real" else "🤖"
    bar = "█" * int(confidence * 20) + "░" * (20 - int(confidence * 20))

    return f"""
{emoji}  {label}
━━━━━━━━━━━━━━━━━━━━━━━━━━━
Confidence:  {confidence:.2%}
[{bar}]

AI Score:    {score:.4f}
Threshold:   0.5000
"""

css = """
body { font-family: 'Segoe UI', sans-serif; }

#title {
    text-align: center;
    color: #ffffff;
    font-size: 2em;
    font-weight: bold;
    padding: 20px;
    background: linear-gradient(135deg, #1a1a2e, #16213e);
    border-radius: 12px;
    margin-bottom: 10px;
}

#subtitle {
    text-align: center;
    color: #aaaaaa;
    margin-bottom: 20px;
}

.gradio-container {
    background-color: #0f0f1a !important;
    max-width: 800px !important;
    margin: auto !important;
}

.upload-box {
    border: 2px dashed #4a4a8a !important;
    border-radius: 12px !important;
    background: #1a1a2e !important;
}

#result {
    background: #1a1a2e !important;
    border-radius: 12px !important;
    font-family: monospace !important;
    font-size: 1.1em !important;
    color: #00ff99 !important;
    padding: 20px !important;
    border: 1px solid #4a4a8a !important;
}

#detect-btn {
    background: linear-gradient(135deg, #4a4aff, #8a2be2) !important;
    border: none !important;
    border-radius: 8px !important;
    color: white !important;
    font-size: 1.1em !important;
    padding: 12px !important;
    cursor: pointer !important;
    width: 100% !important;
}
"""

with gr.Blocks(css=css, theme=gr.themes.Base()) as interface:
    gr.HTML('<div id="title">🔍 AI Image Detector</div>')
    gr.HTML('<div id="subtitle">Upload any image to find out if it was taken by a camera or generated by AI</div>')

    with gr.Row():
        with gr.Column():
            image_input = gr.Image(type="pil", label="Upload Image", elem_classes="upload-box")
            detect_btn  = gr.Button("Detect", elem_id="detect-btn")

        with gr.Column():
            result_output = gr.Textbox(label="Result", lines=8,
                                       elem_id="result", interactive=False)

    detect_btn.click(fn=predict, inputs=image_input, outputs=result_output)

interface.launch()

C:\Users\damax\AppData\Local\Temp\ipykernel_9492\3799031213.py:83: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(css=css, theme=gr.themes.Base()) as interface:


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 93ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
1/1 ━━━━━━━━